In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import os
from matplotlib import colors

## Zadanie 1

### S = 0
### I = 1
### R = 2

In [2]:
N = 60
grid = np.zeros((N,N))
# neighbours = np.zeros((N,N))
grid[30,30] = 1

time = 200
time_tab = [0, 1, 2, 5, 10, 50, 100, 200]
alpha_tab = [0.1, 0.2, 0.3]
beta_tab = [0.1]

In [3]:
def count_infected_neighbors(i,j):
    return (
        grid[(i+1)%N][(j+1)%N] + grid[(i+1)%N][j] + grid[(i+1)%N][j-1] +
        grid[i][(j+1)%N]     +                     grid[i][j-1] +
        grid[i-1][(j+1)%N] + grid[i-1][j] + grid[i-1][j-1]
    )

def step(alpha, beta):
    global grid
    new_grid = grid.copy()

    for i in range(N):
        for j in range(N):

            # I -> R
            if grid[i][j] == 1:
                if np.random.rand() < beta:
                    new_grid[i][j] = 2

            # S -> I
            elif grid[i][j] == 0:
                neighbors = count_infected_neighbors(i, j)
                for _ in range(int(neighbors)):
                    if np.random.rand() < alpha:
                        new_grid[i][j] = 1
                        break

    grid = new_grid


In [4]:
beta = beta_tab[0]

for alpha in alpha_tab:
    os.makedirs(f'zad_1/alpha_{alpha}', exist_ok=True)

    grid = np.zeros((N,N))
    grid[30,30] = 1

    for t in range(101):
        if t in time_tab:
            fig, ax = plt.subplots(figsize=(6,6))

            cmap = colors.ListedColormap(['blue', 'red', 'green'])
            bounds = [0,1,2,3]
            norm = colors.BoundaryNorm(bounds, cmap.N)

            im = ax.imshow(grid, cmap=cmap, norm=norm)

            cbar = plt.colorbar(im, ax=ax, ticks=[0.5,1.5,2.5])
            cbar.ax.set_yticklabels(['Susceptible', 'Infected', 'Recovered'])

            ax.set_title(f't={t}, alpha={alpha}')
            ax.set_xticks([])
            ax.set_yticks([])
            plt.savefig(f'zad_1/alpha_{alpha}/frame_{t}.png', bbox_inches='tight')
            plt.close(fig)

        step(alpha, beta)

In [5]:
# animacja

# grid = np.zeros((N,N))
# grid[30,30] = 1

# fig, ax = plt.subplots()
# im = ax.imshow(grid, cmap='viridis')
# plt.axis('off')

# def animate(i):
#     step(0.3, 0.1)
#     im.set_data(grid)
#     ax.set_title(f'step={i}')
#     return im

# ani = animation.FuncAnimation(fig, animate, frames=100, interval=50)
# ani.save('sir.gif', writer=animation.PillowWriter(fps=10))
# plt.close()

## Zadanie 2

In [15]:
alpha_tab = np.arange(0.1, 1.0, 0.1)

def count_state(val):
    return np.sum(grid == val)

S_all = {}
I_all = {}
R_all = {}

for alpha in alpha_tab:
    grid = np.zeros((N,N))
    grid[30,30] = 1

    # liczba agentów w kolejnych krokach czasowych 
    S = [count_state(0)]
    I = [count_state(1)]
    R = [count_state(2)]

    for t in range(time):
        step(alpha, beta)
        S.append(count_state(0))
        I.append(count_state(1))
        R.append(count_state(2))

    S_all[alpha] = S
    I_all[alpha] = I
    R_all[alpha] = R

In [16]:
os.makedirs('zad_2', exist_ok=True)

for a in alpha_tab:
    plt.figure(figsize=(10,6))
    plt.plot(S_all[a], label='S')
    plt.plot(I_all[a], label='I')
    plt.plot(R_all[a], label='R')

    plt.legend()
    plt.title(f'alpha = {a:.1f}, lambda = {a/(8*beta):.2f}')
    plt.xlabel('czas')
    plt.ylabel('liczba agentów')

    plt.savefig(f'zad_2/alpha_{a:.1f}.png', bbox_inches='tight')
    plt.close()


## Zadanie 3

In [ ]:
os.makedirs('zad_3', exist_ok=True)

omega = N * N # rozmiar siatki

plt.figure(figsize=(10,6))
for alpha in alpha_tab:
    pdf = [I_all[alpha][t] / omega for t in range(len(I_all[alpha]))]
    plt.plot(pdf, label=f'alpha = {alpha:.1f}')

plt.title(f'PDF; beta = {beta}')
plt.xlabel('czas')
plt.ylabel('n(I)')
plt.legend()
plt.savefig(f'zad_3/pdf_I.png', bbox_inches='tight')
plt.close()